# Imports

In [1]:
# Imports

# General imports
import numpy as np
import re
import pandas as pd

# Pytorch and transformers (for LLM)
import transformers, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoModel
transformers.logging.set_verbosity_info()

# For loading documents from a path
from pathlib import Path

# For the embedding module
from sentence_transformers import SentenceTransformer

# %%

# Load device

if torch.backends.mps.is_available():
    # MPS is the GPU model in Mac technology
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device =torch.device("cpu")

print (torch.ones(1, device=device))

tensor([1.], device='cuda:0')


# The different Modules of the RAG

## The foundation model

We are going to work with Foundation Models, that is models that have been pre-trained on large data sets; but we want to work with OpenSource and hosted models. To do so we are going to fetch models from the [HuggingFace plateform](https://huggingface.co/)

In [2]:
class FoundationModel():

    def __init__(self,FOUND_MODEL_PATH,TEMPERATURE=None,MAX_NEW_TOKENS=10000):

        self.model=AutoModelForCausalLM.from_pretrained(FOUND_MODEL_PATH,
                                             #device_map=mps_device,
                                             #device_map=cuda,
                                             torch_dtype="auto",
                                             trust_remote_code=True,
                                             ).to(device)


        self.tokenizer= AutoTokenizer.from_pretrained(FOUND_MODEL_PATH)

        self.model.generation_config.temperature=TEMPERATURE # Config of the temperature
        self.model.generation_config.top_p=None              # Config parameter related to the type of generation (like greedy decoding for instance)

        self.llm = pipeline("text-generation",
                     model=self.model,
                     tokenizer=self.tokenizer,
                     return_full_text=False,
                     max_new_tokens=MAX_NEW_TOKENS,
                     do_sample=True
                     )

        self.num_parameters = self.model.num_parameters()

        print('Number of parameters in my model','{:.2e}'.format(self.num_parameters))


    def generate_response(self,prompt):

        messages = [
            {'role':'user', 'content':prompt}
            ]

        output=self.llm(messages)
        # Note that the output is a list of len 1 which is a dict with key 'generated_text'
        return output


    # We anticipate the use of RAG and create a generate response taking into account the context

    def generate_response_with_context(self, prompt, context):

        # The context is a list of str

        messages = []

        if context:
            for i, ctx in enumerate(context):
                messages.append({'role': 'system','content': f"context {i+1}: {ctx}"})

        messages.append({'role': 'user', 'content': prompt})


        output=self.llm(messages)
        return output

As you will see output of the models will be a list on len 1 in dict format with key ```generated_text```. We somehow reformat the output.

In [3]:
def extract_response(output):
    # output is a list of len 1 as a dict with key 'generated_text'

    return output[0]['generated_text']

Another component of the answer is the inclusion of a reasoning component (which is identified with tags ```<think> ...</think>```). We write which only extract the answer.

In [4]:
def short_response(output):

    response=extract_response(output=output)
    #text = "Before <think>to delete</think> After"
    short = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    return short.strip()

### Unit test for the foundation model

In [5]:
# Unit test
# Start with a model
# Here we list some models and choose a small model Qwen 0.6

Path_SDS="HuggingFaceTB/SmolLM3-3B"
Path_Qwen_4B = "Qwen/Qwen3-4B"
Path_DSR1="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
Path_Q_06="Qwen/Qwen3-0.6B"

f_model = FoundationModel(FOUND_MODEL_PATH=Path_Q_06)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/model.safetensors
Will use dtype=torch.bfloat16 as defined in model's config object
Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645
}



...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Device set to use cuda:0


Number of parameters in my model 5.96e+08


Have a look at the different elements while loading the model; do you recognize some constitutive elements of Transformers ? Look also the number of parameters of the model.

Try several prompts, look at the type of the output directly from the llm or simply the output produced with `generate_response`methods (standard or short).

In [6]:
# Launch a query
f_model.generate_response("Tu vas bien?")
# print the output and look to the format

[{'generated_text': '<think>\nOkay, the user just asked, "Tu vas bien?" which translates to "Are you doing well?" in French. I need to respond in French, but I should check for any possible errors in the translation. The user might be testing if I can understand the question correctly. Let me make sure the response is polite and in line with French expressions. I should acknowledge their question and offer further assistance if needed. Also, I should keep the tone friendly and helpful.\n</think>\n\nBonjour! Je suis bien, merci! Si tu as des questions ou as besoin de quelque aide, n\'hésite pas à me faire savoir. 😊'}]

We are going to study a specific example. Generate and print the answer to the query below. Which natural remark once have to keep in mind while using Open Source models ?

In [7]:
# A specific prompt

prompt_RF='Is Robert Redford alive ?; Answer must be [Yes] or [No]'
f_model.generate_response(prompt_RF)

[{'generated_text': "<think>\nOkay, so the user is asking if Robert Redford is alive. They want a straightforward answer, either [Yes] or [No]. Let me think.\n\nFirst, I need to confirm his current status. Robert Redford passed away on January 14, 2022, at 10:46 AM in his home in New York, New York. That's in 2022, not 2021. So, the answer should be [No], because he is deceased.\n\nWait, but sometimes people might confuse the date. Let me double-check. Yes, his death was in 2022, so he's no longer alive. I should make sure there's no other recent information I'm missing. No, I don't think so. The answer is definitely [No].\n</think>\n\n[No]"}]

In [9]:
%load solutions/specific_query.py

ValueError: 'solutions/specific_query.py' was not found in history, as a file, url, nor in the user namespace.

## The Embedding model

Our foundation model will be mostly used as a decoder, even though it includes an encoder one usually chooses a separate embedding model. Indeed the embedding model of the decoder is designed for the generation process but fails to produce most adequate embeddings for a semantic study (like computing similarity). We make use here of an Open Source embedding model from Library `SentenceTransformer` (in particular we will create embeddings for sentences or more precisely for *chunks*).

In [10]:
class EmbeddingModel():

    def __init__(self,EMBEDD_MODEL_PATH):

        # EMBEDD_MODEL_PATH is the name of the embedding model used within the SentenceTransformer lib

        self.Embedmodel=SentenceTransformer(EMBEDD_MODEL_PATH).to(device)
        self.dim=SentenceTransformer(EMBEDD_MODEL_PATH).get_sentence_embedding_dimension()


    def get_embeddings(self,texts):

        # texts is a list of strings (which is supposed to be the list of chinks; without the source)
        # we return embeddings of torch type with shape (len(texts),self.dim)

        embeddings=self.Embedmodel.encode(texts,convert_to_tensor=True,normalize_embeddings=True).to(device)
        return embeddings


    def compute_cos_sim_embed(self,embed1,embed2):

        # embed1,embeds2 are two embeddings of shape (1,dim)
        # We compute the cos-similarity of two texts (it is returned as a float)

        embed1=embed1.view(-1)
        embed2=embed2.view(-1)

        norm1=torch.norm(embed1,p=2,dim=0)
        norm2=torch.norm(embed2,p=2,dim=0)

        scal = torch.dot(embed1,embed2)

        return scal.item()/(norm1.item()*norm2.item())


    def compute_cos_sim_texts(self,text_1,text_2):

         # text1,text2 are two str
        # We compute the cos-similarity of two texts (it is returned as a float)

        embeds = self.get_embeddings(texts=[text_1,text_2])

        return self.compute_cos_sim_embed(embeds[0],embeds[1])


### Unit test for the embedding model

In [11]:
# Unit test

Embed_mini="all-MiniLM-L6-v2"
EmbedModel=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)

# Once again have a look at the parameters of the model.

sentences=['Hello World','How is the weather ?']

embeddings = EmbedModel.get_embeddings(texts=sentences)

print(type(embeddings),embeddings.shape,type(embeddings[0].dtype))

em = EmbedModel.compute_cos_sim_embed(embeddings[0],embeddings[1])

print(em)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

loading file vocab.txt from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/vocab.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/special_tokens_map.json
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
loading file chat_template.jinja from cache at None


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/model.safete

<class 'torch.Tensor'> torch.Size([2, 384]) <class 'torch.dtype'>
0.30355432629585266


## Notion of Chunk and Splitter

We decide to create a class for managing chunks. A *chunk* will be defined by:

-  ```source``` the name of the .txt file from which the chunk comes from
-  ```content``` which is the str that composes the chunk
-  ```embedding``` which is the embedding associated to ```content```.

In [12]:
class Chunk():

    def __init__(self,source,content,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        #dim is the common dimension of the embeddings
        dim = self.embedding_model.dim

        # A chunk is defined by its source (str); its content (str); its embedding (a torch which shape (1,dim))

        self.source=str(source)
        self.content=str(content)
        self.embedding=self.embedding_model.get_embeddings(texts=[content]).reshape(1,dim)


    def print_chunk(self):

        print('source:',self.source,'content:',self.content,'embedding shape:',self.embedding.shape)

## Todwards the index: the splitter

We are going to chunk resources and then compare the similarity of each of these chunks with the one of the query. We follow the plan of the lecture and start with the splitter.

This class aims in getting the documents that will be in a folder with address ```path_doc```, and return as an output of the method ```get_chunks``` the chunks associated to these resources. Note that we keep track of which document is issued each chunk (via the ```source``` feature).

In [13]:
class Splitter():

    def __init__(self,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        self.docs = []
        # We store the original documents as a list of .txt files (format is {"source":'File_name',"content_page":(str)})
        self.chunks=[]
        # This will be the list of chunks

    def get_documents(self,path_doc):
        # PATH_DOC is the Path form where the documents will be found (each document is a.txt file).
        docs=[]

        for file in Path(path_doc).rglob("*.txt"):
            name=file.name
            with open(file, "r", encoding="utf-8") as file:
                resource=file.read().strip()
                if resource:
                    #print(resource,len(resource))
                    docs.append({"source":name,"content_page":resource})

        self.docs=docs


    def get_chunks_contents_from_1_doc(self,file_name,content_page,chunk_size,overlap,sentence_split=False):

        if chunk_size < overlap:
            raise Exception('Careful overlap must be smaller than chunk_size')

        # Now we chunk according to chunk size and overlap

        if sentence_split:

            content=content_page.split(".")

            for text in content:

                text = text.lstrip()

                if not text=="":
                    self.chunks.append(Chunk(source=file_name,
                      content=text,embed_model=self.embedding_model))

        else:

            current = 0

            while current < len(content_page):
                end = min(len(content_page),current+chunk_size)
                content = content_page[current:end]

                self.chunks.append(Chunk(source=file_name,
                      content=content,embed_model=self.embedding_model))

                current += chunk_size - overlap


    def get_chunks(self,path_doc,chunk_size,overlap,sentence_split=False):

        self.get_documents(path_doc=path_doc)

        docs=self.docs

        for doc in docs:

            self.get_chunks_contents_from_1_doc(file_name=doc["source"],
                                                content_page=doc["content_page"],
                                                chunk_size=chunk_size,
                                                overlap=overlap,
                                                sentence_split=sentence_split)

    def reset_splitter(self):

        self.docs=[]
        self.chunks=[]


### Unit test for the splitting module

In [17]:
# Unit test

this_path = Path.cwd()/"Docs"
embed_model=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)
Split=Splitter(embed_model)

Split.reset_splitter()
Split.get_documents(path_doc=this_path)
Split.get_chunks(path_doc=this_path,chunk_size=30,overlap=15,sentence_split=True)

print(len(Split.chunks))

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/model.safete

2


In [ ]:
# Do your own tests, with overlaps, without, what is the impact ? like for example :


Robert_Redford.txt 
 Robert Redford passed away last month 
 torch.Size([1, 384])
Personalities.txt 
 Albert Einstein proposed the theory of relativity, which transformed our understanding of time,space, and gravity 
 torch.Size([1, 384])
Personalities.txt 
 Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity and won two Nobel Prizes 
 torch.Size([1, 384])
Personalities.txt 
 Isaac Newton formulated the laws of motion and universal gravitation, laying the foundation for classical mechanics 
 torch.Size([1, 384])
Personalities.txt 
 Charles Darwin introduced the theory of evolution by natural selection in his book 'On the Origin of Species' 
 torch.Size([1, 384])
Personalities.txt 
 Ada Lovelace is regarded as the first computer programmer for her work on Charles Babbage's early mechanical computer, the Analytical Engine 
 torch.Size([1, 384])


## Index, Database and Retriever

We are now in position to define the retriever. In our setting with very few document we will build our index (when the number of resources is pretty high (over $10^4$) we make use of an index (which is trained for that purpose); one can for example make use of the FAISS library.

In [15]:
class Retriever():

    def __init__(self,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        # The index is a list of (Id(int),chunk); chunk needs the size DIM for the Embeddings
        self.index=[]


    def add_elements_to_index(self,chunks):

        # chunks is a list of chunk

        num = len(self.index)

        for chunk in chunks:

            self.index.append([num,chunk])
            num+=1

    def search_best(self,query,number_of_hits=3,adapt=False):
      query_embed = self.embedding_model.get_embeddings(texts=[query]).to(device).reshape(1,self.embedding_model.dim)

      results=[]

      index=self.index

      scores=[]

      for id, chunk in self.index:

        sim = self.embedding_model.compute_cos_sim_embed(query_embed,chunk.embedding)
        scores.append((id, chunk, sim))

      results=sorted(scores,key=lambda x:x[2],reverse=True)[:min(number_of_hits,len(index))]

      if adapt:

            i=1
            go=True
            while go and i<len(results):
                if results[i][2] < results[i-1][2]*0.5:
                    go=False
                else:
                    i+=1

            results=results[:i]

      return result

    def reset_Retriever_index(self):

        self.index=[]

In [ ]:
# %load solutions/retriever_class.py

### Unit test of the Retriever module

In [18]:
# Unit test

# Recall that we have already in our previous unit tests defined an embedding model and a splitter

        # embed_model=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)

        # Split=Splitter(embed_model)
        # Split.chunks=[]
        # Split.get_chunks(path_doc=this_path,chunk_size=30,overlap=15,sentence_split=True)

print(Split.docs)

chunks=Split.chunks

print(len(chunks))

retriever = Retriever(embed_model)

# Add the chunks to the index

# Get the best results using your retriever to the query

        #prompt_RF='Is Robert Redford alive ?; Answer must be [Yes] or [No]'

[{'source': 'Personalities.txt', 'content_page': 'Albert Einstein proposed the theory of relativity, which transformed our understanding of time,space, and gravity\n Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity and won two Nobel Prizes'}, {'source': 'Robert_Redford.txt', 'content_page': 'Robert Redford passed away last month'}]
2


In [ ]:
# %load solutions/retriever_specific_prompt.py

# The RAG Architecture

In [19]:
class RAG():

    def __init__(self,CONFIG):

        self.foundation_model=FoundationModel(FOUND_MODEL_PATH=CONFIG['FOUND_MODEL_PATH'])
        self.Embedding_model=EmbeddingModel(EMBEDD_MODEL_PATH=CONFIG['EMBEDD_MODEL_PATH'])
        self.splitter=Splitter(self.Embedding_model)
        self.retriever=Retriever(self.Embedding_model)

        self.dim_embed = CONFIG['DIM_EMBED']
        self.chunk_size = CONFIG['CHUNK_SIZE']
        self.overlap = CONFIG['OVERLAP']


    def reset_index(self):

        self.retriever.reset_Retriever_index()
        self.splitter.reset_splitter()


    def load_documents_and_get_chunks(self,path,sentence_split=False):

        self.splitter.get_chunks(path_doc=path,
                                 chunk_size=self.chunk_size,
                                 overlap=self.overlap,
                                 sentence_split=sentence_split)

        chunks = self.splitter.chunks

        self.retriever.add_elements_to_index(chunks=chunks)


    def get_retrieval(self,query,number_of_hits):

        retrieved_info = self.retriever.search_best(query=query,number_of_hits=number_of_hits)

        # It is the full information of the form (Id, chunk, sim)

        retrieved=[]

        for elem in retrieved_info:

            i,chunk, distance=elem

            retrieved.append(chunk.content)

        # We get rid of repeated items
        return list(dict.fromkeys(retrieved))

    def generate_response_rag(self,query):

        retrieved=self.get_retrieval(query=query,
                                          number_of_hits=3)

        return self.foundation_model.generate_response_with_context(prompt=query,
                                                                   context=retrieved)


# Experiments

In [20]:
CONFIG = {
    'FOUND_MODEL_PATH':Path_Q_06,
    #'FOUND_MODEL_PATH':"Qwen/Qwen3-4B",
    'EMBEDD_MODEL_PATH':"all-MiniLM-L6-v2",
    'DIM_EMBED':384,
    'CHUNK_SIZE':300,
    'OVERLAP':30
        }

rag = RAG(CONFIG=CONFIG)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention

Number of parameters in my model 5.96e+08


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/model.safete

In [21]:
# We initialize our RAG with our documents

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs",sentence_split=True)

chunks=rag.splitter.chunks

## The Robert Redford experiment

In [24]:
# Once again for the prompt below compare the answer with and without RAG
prompt_RF='Is Robert Redford alive ?; Answer must be [Yes] or [No]'

context = rag.get_retrieval(query=prompt_RF, number_of_hits=3)

response_RAG = rag.foundation_model.generate_response_rag(prompt=prompt_RF)

print(response_RAG)
# Print what is the retrieved context

TypeError: 'Chunk' object is not subscriptable

In [23]:
%load solutions/experiment_RF.py

ValueError: 'solutions/experiment_RF.py' was not found in history, as a file, url, nor in the user namespace.

### Reranking

You noticed that since you asked for a fixed ```number_of_hits``` some of the context might be pointless. We aim in preventing this and selecting the retrieved context. We provide two ways for that.

**1st we plot the similarity and try to find an accurate split out of it.**

In [ ]:
# In both methods we start with our retrieved information

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs",sentence_split=True)

retrieved_info = rag.retriever.search_best(query=query,number_of_hits=3)
# It is (Id,chunk,sim) and it is ordered by sim


In [ ]:
data=[]
select_data=[]

temp_sim = retrieved_info[0][2] # The highest sim

for item in retrieved_info:

    Id,chunk,sim=item
    data.append((Id,chunk.content,sim))

    temp = temp_sim*0.9 # the current sim - 10%

    if sim > temp:

        temp_sim = sim
        select_data.append((Id,chunk.content,sim))


data = pd.DataFrame(data,columns=["Id","chunk","sim"])
selected_data = pd.DataFrame(select_data,columns=["Id","chunk","sim"])

print(data)
print(selected_data)

   Id                                              chunk       sim
0   0              Robert Redford passed away last month  0.704102
1   1  Albert Einstein proposed the theory of relativ...  0.124124
2   3  Isaac Newton formulated the laws of motion and...  0.050023
   Id                                  chunk       sim
0   0  Robert Redford passed away last month  0.704102


**2nd method : *LLM as a judge***

This time we give the query and the retrieved information to a LLM called *judge (as a) LLM* and ask it to select the more suitable chunks.

In [ ]:
# We define the judge (it is always better to take another LLM and to do the opposite of what we are doing now: usually one chooses the judge to be a more powerful model; here for VRAM reasons we select a weaker one).DS_Store

judge_llm = FoundationModel(FOUND_MODEL_PATH=Path_SDS)


loading configuration file config.json from cache at /Users/reveilla/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM3-3B/snapshots/a07cc9a04f16550a088caea529712d1d335b0ac1/config.json
Model config SmolLM3Config {
  "architectures": [
    "SmolLM3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128012,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atte

Number of parameters in my model 3.08e+09


In [ ]:
context_chunks = data["chunk"].tolist()

#print(context_chunks)

#prompt = print(f"Given [{query}] tell me among the elements of [{context_chunks}] which ones are the more relevant to [{query}]; give me the answer as a list of elements of [{context_chunks}]")

prompt=f""" You are a clever assistant. Given the query: "{query}"; here is a list of resources "{context_chunks}". For each of those say which is useful to answer the query. Give your result by only listing the useful results.
"""

print(judge_llm.generate_response(prompt=prompt))

[{'generated_text': '<think>\nOkay, let\'s see. The user is asking if Robert Redford is alive. The provided resources are three sentences. I need to check each one to see if they answer the question.\n\nFirst resource: "Robert Redford passed away last month." That directly states he died, so it\'s useful because it provides a clear answer of No.\n\nSecond resource: "Albert Einstein proposed the theory of relativity..." This is about Einstein\'s work, which is irrelevant to Robert Redford\'s current state. So not useful here.\n\nThird resource: "Isaac Newton formulated the laws of motion and universal gravitation..." Again, about Newton\'s contributions to physics. Not related to Robert Redford\'s life status. So not useful either.\n\nOnly the first resource answers the question. The others are about different people and topics. Therefore, the useful answer is [No], from the first statement.\n</think>\n\n[No]  \n[\'Robert Redford passed away last month\']'}]


## The personalities experiment

In [ ]:
# You can extend the study by now looking to questions that still address one chunk but which are related semantically.
# We introduce 5 queries; each one is related to one and only one personality

queries = [
"Who introduced the theory of relativity?",
"Who was the first computer programmer?",
"What did Isaac Newton contribute to science?",
"Who won two Nobel Prizes for research on radioactivity?",
"What is the theory of evolution by natural selection?"
]

print(queries)

['Who introduced the theory of relativity?', 'Who was the first computer programmer?', 'What did Isaac Newton contribute to science?', 'Who won two Nobel Prizes for research on radioactivity?', 'What is the theory of evolution by natural selection?']
